In [24]:
import fredapi
from dotenv import load_dotenv
import os
import pandas as pd
import sqlalchemy as db
from sqlalchemy import text, column

In [25]:
load_dotenv()
api_key = os.getenv("API_KEY")

In [26]:
fred = fredapi.Fred(api_key)

In [27]:
data = fred.get_series('IRLTLT01USM156N')

In [28]:
data_vintage = fred.get_series_vintage_dates('IRLTLT01USM156N')

In [29]:
pd.to_datetime(data_vintage)

DatetimeIndex(['2013-06-03', '2013-07-01', '2013-08-01', '2013-08-21',
               '2013-10-01', '2013-11-01', '2013-12-02', '2014-02-03',
               '2014-11-03', '2014-12-01',
               ...
               '2025-05-15', '2025-06-16', '2025-07-15', '2025-08-15',
               '2025-09-15', '2025-10-15', '2025-11-17', '2025-12-15',
               '2026-01-15', '2026-02-16'],
              dtype='datetime64[us]', length=135, freq=None)

In [30]:
data = data.dropna()

In [31]:
data.head()

1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
dtype: float64

In [32]:
data.tail()

2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
dtype: float64

In [33]:
data.describe

<bound method NDFrame.describe of 1953-04-01    2.83
1953-05-01    3.05
1953-06-01    3.11
1953-07-01    2.93
1953-08-01    2.95
              ... 
2025-09-01    4.12
2025-10-01    4.06
2025-11-01    4.09
2025-12-01    4.14
2026-01-01    4.21
Length: 874, dtype: float64>

In [34]:
type(data)

pandas.Series

In [35]:
print(fred.get_series_all_releases('IRLTLT01USM156N'))

           realtime_start                 date value
0     2024-04-10 00:00:00  1953-04-01 00:00:00  2.83
1     2024-04-10 00:00:00  1953-05-01 00:00:00  3.05
2     2024-04-10 00:00:00  1953-06-01 00:00:00  3.11
3     2024-04-10 00:00:00  1953-07-01 00:00:00  2.93
4     2024-04-10 00:00:00  1953-08-01 00:00:00  2.95
...                   ...                  ...   ...
2070  2025-10-15 00:00:00  2025-09-01 00:00:00  4.12
2071  2025-11-17 00:00:00  2025-10-01 00:00:00  4.06
2072  2025-12-15 00:00:00  2025-11-01 00:00:00  4.09
2073  2026-01-15 00:00:00  2025-12-01 00:00:00  4.14
2074  2026-02-16 00:00:00  2026-01-01 00:00:00  4.21

[2075 rows x 3 columns]


In [36]:
# Approximate fixed lags in days from value date to publication as FREDAPI doesn't offers real publish dates for old publications
rate_lags = {
    'UNRATE': 45,  # ~6 weeks after reference month end
    'CPIAUCSL': 15,  # ~2 weeks after reference month end
    'M2SL': 30,
    'WALCL': 7,
    'NFCI': 7,
    'ICSA': 5,  # Thursdays, covers prior week
}

def get_fred_data(serie):
    '''
    Process and store in the db the indicator
    '''
    #get the data
    data = fred.get_series(serie)
    #drop nan values
    data = data.dropna()

    if serie in rate_lags:
        data.index = data.index + pd.DateOffset(days=rate_lags[serie])

    # convert Series to DataFrame
    data = data.reset_index()
    data.columns = ['date', 'value']

    # remove timezone info
    data['date'] = pd.to_datetime(data['date']).dt.tz_localize(None)

    # add series name
    data['serie'] = serie

    return data

get_fred_data('FEDFUNDS')

,date,value,serie
0,1954-07-01,0.80,FEDFUNDS
1,1954-08-01,1.22,FEDFUNDS
2,1954-09-01,1.07,FEDFUNDS
3,1954-10-01,0.85,FEDFUNDS
4,1954-11-01,0.83,FEDFUNDS
...,...,...,...
854,2025-09-01,4.22,FEDFUNDS
855,2025-10-01,4.09,FEDFUNDS
856,2025-11-01,3.88,FEDFUNDS
857,2025-12-01,3.72,FEDFUNDS


In [38]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
from bs4 import BeautifulSoup
import time
from datetime import datetime

In [127]:
driver = webdriver.Chrome()
driver.get("https://www.forexfactory.com/calendar")

engine = db.create_engine('sqlite:///../data/data.db')

year = 2007 #this is the first year
year_now = datetime.now().year
month = None
last_date = None

with engine.connect() as conn:
    result = conn.execute(text(f'SELECT MAX(date) FROM Events'))
    last_date = result.scalar() #gets last date if exists

if last_date is not None:
    year = pd.to_datetime(last_date).year
    month = pd.to_datetime(last_date).month

years = range(year, year_now + 1)
 #months for data search
months = {'Jan': 31,
              'Feb': 28,
              'Mar': 31,
              'Apr': 30,
              'May': 31,
              'Jun': 30,
              'Jul': 31,
              'Aug': 31,
              'Sep': 30,
              'Oct': 31,
              'Nov': 30,
              'Dec': 31}

if month is not None:
     month, day = list(months.items())[month]
     months = {month: day} #rescribe the dict to get only the month we need

macro_events = pd.DataFrame()

#search the filter button
elements = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[1]/ul/li[8]/a')
elements.click()
#deselect the countries
deselect_all = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div[2]/p/span/a[2]')
deselect_all.click()
#select only usa
usa_element = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div[2]/div/div[2]/div[5]/div[1]')
usa_element.click()
#apply changes
apply_changes = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[2]/div/table/tbody/tr/td[3]/input[1]')
apply_changes.click()
time.sleep(1)

for year in years:
    for month, last_day in months.items():

        #search te calendar
        calendar_element = driver.find_element(By.CLASS_NAME, 'calendar__options.left')
        calendar_element.click()
        #select date range
        date_range_element = driver.find_element(By.ID, 'calendar-date-range-1')
        date_range_element.click()
        date_range_element.clear()
        date_range_element.send_keys(f'{month} 1, {year} – {month} {last_day}, {year}')
        #apply date settings
        apply_date = driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[2]/div[1]/div/table/tbody/tr/td[2]/input[1]')
        apply_date.click()
        time.sleep(2)

        #scroll page
        actions = ActionChains(driver)
        actions.send_keys(Keys.END).perform()
        time.sleep(1)

        #get html
        page_html = driver.page_source
        #read the html with beautifulsoup
        soup = BeautifulSoup(page_html, 'lxml')

        #get the event table
        tables = soup.find_all('table', {'class': 'calendar__table'})
        rows = soup.find_all('tr', {'class': 'calendar__row'})

        day = ""
        event = ""
        actual = ""
        previous = ""
        seen = set()
        #creates the empty dataset
        data = []

        for row in rows:
            date_span = row.find('span', {'class': 'date'})
            event_span = row.find('span', {'class': 'calendar__event-title'})
            actual_span = row.find('td', {'class': 'calendar__actual'})
            previous_span = row.find('td', {'class': 'calendar__previous'})

            if date_span is not None:
                day = date_span.text.strip()
                event = ''
                actual = ''
                previous = ''

            if event_span is not None:
                event = event_span.text.strip()
                actual = ''  # reset actual when we find a new event
                previous = '' #reset previous too

            if actual_span is not None:
                actual = actual_span.text.strip()

            if previous_span is not None:
                previous = previous_span.text.strip()

            # Only print when we have a full, new (day, event) combo
            if day and event:
                key = (day, event)
                if key not in seen:
                    seen.add(key)
                    #format the date
                    date = f'{day} {year}'
                    date = pd.to_datetime(date, format='%a %b %d %Y')
                    #add data to dataset

                    data.append({
                        "date": date,
                        "event": event,
                        "actual": actual,
                        "previous": previous
                    })
        df = pd.DataFrame(data)

        events = [
            #monetary policy
            'Federal Funds Rate',
            'FOMC Statement',
            'FOMC Press Conference',
            'FOMC Economic Projections',
            'Fed Chair Press Conference',

            #inflation
            'Core CPI m/m',
            'CPI m/m',
            'Core CPI y/y',
            'PPI m/m',
            'Core PCE Price Index m/m',
            'PCE Price Index m/m',

            #laboral
            'Non-Farm Employment Change',
            'Unemployment Rate',
            'Average Hourly Earnings m/m',
            'Initial Jobless Claims',

            #growth
            'Advance GDP q/q',
            'Prelim GDP q/q',
            'Final GDP q/q',

            #Spending
            'Retail Sales m/m',
            'Core Retail Sales m/m',
            'Personal Spending m/m',
            'Personal Income m/m',

            #Economic feeling
            'ISM Manufacturing PMI',
            'ISM Services PMI',
            'S&P Global Manufacturing PMI',
            'S&P Global Services PMI',

            #Housing
            'Building Permits',
            'Housing Starts',
            'Existing Home Sales',
            'New Home Sales',
        ]


        df = df[df['event'].isin(events)]

        print(f'{len(df)} new entries added')
        macro_events = pd.concat([macro_events, df])

        actions.send_keys(Keys.HOME).perform()
        time.sleep(2)

    #saves this into the db
with engine.connect() as conn:
    macro_events.to_sql('Events', con=conn, if_exists='append', index=False)
    conn.commit()
    print(f'{len(macro_events)} new entries added to the database')

driver.quit()

22 new entries added
22 new entries added to the database


In [40]:
type(day)
print(day)

Wed Jan 31


In [87]:
macro_events['date'].iloc[1].month

1

In [2]:
import pandas as pd
import sqlalchemy as db

engine = db.create_engine('sqlite:///../data/data.db')

with engine.connect() as conn:
    tray = pd.read_sql('SELECT * FROM Events', con=conn)

tray.tail()

,date,event,actual,previous
2037,2026-07-29 00:00:00.000000,FOMC Press Conference,,
2038,2026-07-30 00:00:00.000000,Advance GDP q/q,,
2039,2026-07-30 00:00:00.000000,Core PCE Price Index m/m,,
2040,2026-07-30 00:00:00.000000,Personal Income m/m,,
2041,2026-07-30 00:00:00.000000,Personal Spending m/m,,


In [130]:
len(tray)

478

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import pandas as pd
from bs4 import BeautifulSoup
import time
from selenium.common.exceptions import NoSuchElementException
import sqlalchemy as db
from sqlalchemy import text

In [10]:
def search_event():
    events = [
                #monetary policy
                'US Federal Funds Rate',
                'US FOMC Statement',
                'US FOMC Press Conference',
                'US FOMC Economic Projections',


                #inflation
                'US Core CPI m/m',
                'US CPI m/m',
                'US CPI y/y',
                'US PPI m/m',
                'US Core PCE Price Index m/m',

                #laboral
                'US Non-Farm Employment Change',
                'US Unemployment Rate',
                'US Average Hourly Earnings m/m',

                #growth
                'US Advance GDP q/q',
                'US Prelim GDP q/q',
                'US Final GDP q/q',

                #Spending
                'US Retail Sales m/m',
                'US Core Retail Sales m/m',
                'US Personal Spending m/m',
                'US Personal Income m/m',

                #Economic feeling
                'US ISM Manufacturing PMI',
                'US ISM Services PMI',

                #Housing
                'US Building Permits',
                'US Housing Starts',
                'US Existing Home Sales',
                'US New Home Sales',
            ]

    def select_event(event, driver):

        actions = ActionChains(driver)

        driver.find_element(By.XPATH, '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[1]/ul/li[7]/a').click()
        time.sleep(2)

        driver.find_element(By.XPATH, '//*[@id="calendar-search-input"]').click()
        time.sleep(2)

        actions.send_keys(event)
        actions.perform()
        time.sleep(2)

        driver.find_element(By.LINK_TEXT, event).click()

        return driver

    def search_dates(driver):
        #search for the dates inside the page loading all the data
        actions = ActionChains(driver)

        while True:
            try:
                #if footer is hidden stops
                driver.find_element(By.CSS_SELECTOR, ".foot.hidden")
                time.sleep(1)
                actions.send_keys(Keys.END).perform()
                time.sleep(1)
                actions.send_keys(Keys.HOME).perform()
                time.sleep(1)
                break

            except NoSuchElementException:
                # if not footer is hidden, click more button
                more = driver.find_element(By.CLASS_NAME, "more")
                more.click()
                time.sleep(1)
        return driver


    def get_events(driver):
        #get html
        page_html = driver.page_source
        print('get html')

        #read the html with beautifulsoup
        soup = BeautifulSoup(page_html, 'lxml')

        #search for the dates, the current value and the previous value if exist
        dates = soup.find_all(class_= 'calendarhistory__row nowrap calendarhistory__row--history')
        currents = soup.find_all(class_= 'calendarhistory__row calendarhistory__row--actual')
        previouses = soup.find_all(class_= 'calendarhistory__row calendarhistory__row--previous nowrap')

        data = {}

        #stores the data into a dict
        for i in range(0, len(dates)):

            date = pd.to_datetime(dates[i].text.strip())

            data[date] = (currents[i].text.strip() if currents else None,
                                           previouses[i].text.strip() if previouses else None)

        print('data obtained')
        driver.close()
        return data


    #starts db engine
    engine = db.create_engine('sqlite:///data.db')

    #option to run navigator silently
    chrome_options = Options()
    #chrome_options.add_argument("--headless=new")

    # forces resolution
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--start-maximized")

    # changes user agent
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    chrome_options.add_argument(f'user-agent={user_agent}')

    # disable some features
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=chrome_options) #delete the option to see the navigator proces
    driver.get("https://www.forexfactory.com/calendar")

    for i in events:
        with engine.connect() as conn:
            result = conn.execute(text((f'SELECT MAX(date) FROM Events WHERE event = "{i}"')))
            last_date = result.scalar()
            if last_date:
                last_date = pd.to_datetime(last_date).normalize()


            print(i)

            data = get_events(search_dates((select_event(i, driver))))

            #saves data into db testing the last date in the db for the event
            with engine.connect() as conn:
                for date, (current, previous) in data.items():
                    if last_date is None or date > last_date:
                        conn.execute(text((f'INSERT INTO Events VALUES ("{str(date)}", "{i}", "{current}", "{previous}")')))
                        conn.commit()
                        print(f'{i} saved')
                print(f'{i} is up to date')
                conn.close()
    print('Everything is up to date')
    driver.quit()

search_event()

Procesando: US Federal Funds Rate
Cerrando navegador...


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x1207dd3
	0x1207e14
	0x1011bee
	0x104f2d5
	0x107e8f6
	0x107a491
	0x1079ab2
	0xfe500e
	0xfe55ae
	0xfe5a8d
	0x1466924
	0x1461bf7
	0x147f5a0
	0x1220f58
	0x122891d
	0xfe4c69
	0xfe42b0
	0x15c876f
	0x750d5d49
	0x7719d81b
	0x7719d7a1


In [52]:
print(data)

None


In [38]:
engine = db.create_engine('sqlite:///../data/raw/data.db')

with engine.connect() as conn:
    conn.execute(text('DELETE FROM Events'))

In [39]:
engine = db.create_engine('sqlite:///../data/raw/data.db')

with engine.connect() as conn:
    data = pd.read_sql('SELECT * FROM Events', conn)
data.head()

,date,event,actual,previous
0,2007-01-03 00:00:00.000000,ISM Manufacturing PMI,51.4,49.5
1,2007-01-04 00:00:00.000000,ISM Services PMI,57.1,58.9
2,2007-01-05 00:00:00.000000,Non-Farm Employment Change,167K,132K
3,2007-01-05 00:00:00.000000,Unemployment Rate,4.5%,4.5%
4,2007-01-05 00:00:00.000000,Average Hourly Earnings m/m,0.5%,0.3%


In [12]:
import time
import random
import pandas as pd
import sqlalchemy as db
from sqlalchemy import text
from bs4 import BeautifulSoup

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ── Helpers ────────────────────────────────────────────────────────────────────

def human_delay(min_s=1.2, max_s=3.5):
    """Pausa aleatoria para simular comportamiento humano."""
    time.sleep(random.uniform(min_s, max_s))


def human_type(actions, text):
    """Escribe carácter a carácter con pausas aleatorias."""
    for char in text:
        actions.send_keys(char)
        actions.perform()
        time.sleep(random.uniform(0.05, 0.18))


def wait_for(driver, by, selector, timeout=15):
    """Espera explícita hasta que un elemento esté clickable."""
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, selector))
    )


# ── Funciones del scraper ──────────────────────────────────────────────────────

def select_event(event, driver):
    """Navega al evento concreto en ForexFactory."""
    actions = ActionChains(driver)

    # Clic en el filtro de categorías (Events)
    wait_for(driver, By.XPATH,
             '//*[@id="content"]/section[2]/div[3]/div/div/div/div/div[1]/ul/li[7]/a').click()
    human_delay(1.5, 3.0)

    # Clic en el buscador
    search_input = wait_for(driver, By.XPATH, '//*[@id="calendar-search-input"]')
    search_input.click()
    human_delay(0.5, 1.2)

    # Escribe el evento de forma humana
    human_type(actions, event)
    human_delay(1.0, 2.0)

    # Clic en el resultado exacto
    wait_for(driver, By.LINK_TEXT, event).click()
    human_delay(2.0, 3.5)

    return driver


def load_all_history(driver):
    """
    Hace scroll para cargar todos los datos históricos.
    Para cuando el footer .foot aparece con clase 'hidden'.
    """
    actions = ActionChains(driver)

    while True:
        try:
            driver.find_element(By.CSS_SELECTOR, ".foot.hidden")
            # Footer oculto → ya cargó todo
            human_delay(0.8, 1.5)
            actions.send_keys(Keys.HOME).perform()
            human_delay(0.5, 1.0)
            break

        except NoSuchElementException:
            # Todavía hay más datos → click en "more"
            try:
                more_btn = driver.find_element(By.CLASS_NAME, "more")
                more_btn.click()
                human_delay(0.8, 1.8)
            except NoSuchElementException:
                # No hay botón "more" ni footer oculto → salimos igualmente
                break

    return driver


def parse_events(driver):
    """
    Extrae fechas, valores actuales y anteriores del HTML de la página.
    Devuelve un dict { datetime: (current, previous) }
    """
    soup = BeautifulSoup(driver.page_source, "lxml")

    dates     = soup.find_all(class_="calendarhistory__row nowrap calendarhistory__row--history")
    currents  = soup.find_all(class_="calendarhistory__row calendarhistory__row--actual")
    previouses = soup.find_all(class_="calendarhistory__row calendarhistory__row--previous nowrap")

    data = {}
    for i in range(len(dates)):
        try:
            date = pd.to_datetime(dates[i].text.strip())
        except Exception:
            continue

        current  = currents[i].text.strip()  if i < len(currents)   else None
        previous = previouses[i].text.strip() if i < len(previouses) else None

        data[date] = (current, previous)

    print(f"  → {len(data)} registros encontrados")
    return data


# ── Función principal ──────────────────────────────────────────────────────────

def search_event():
    events = [
        # Política monetaria
        "US Federal Funds Rate",
        "US FOMC Statement",
        "US FOMC Press Conference",
        "US FOMC Economic Projections",

        # Inflación
        "US Core CPI m/m",
        "US CPI m/m",
        "US CPI y/y",
        "US PPI m/m",
        "US Core PCE Price Index m/m",

        # Laboral
        "US Non-Farm Employment Change",
        "US Unemployment Rate",
        "US Average Hourly Earnings m/m",

        # Crecimiento
        "US Advance GDP q/q",
        "US Prelim GDP q/q",
        "US Final GDP q/q",

        # Gasto
        "US Retail Sales m/m",
        "US Core Retail Sales m/m",
        "US Personal Spending m/m",
        "US Personal Income m/m",

        # Sentimiento económico
        "US ISM Manufacturing PMI",
        "US ISM Services PMI",

        # Vivienda
        "US Building Permits",
        "US Housing Starts",
        "US Existing Home Sales",
        "US New Home Sales",
    ]

    # ── Base de datos ──────────────────────────────────────────────────────────
    engine = db.create_engine("sqlite:///data.db")

    # Crea la tabla si no existe
    with engine.connect() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS Events (
                date     TEXT,
                event    TEXT,
                current  TEXT,
                previous TEXT,
                PRIMARY KEY (date, event)
            )
        """))
        conn.commit()

    # ── Driver con undetected_chromedriver ─────────────────────────────────────
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = uc.Chrome(options=options, headless=False, version_main=145)

    # Elimina trazas extra de webdriver
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    try:
        driver.get("https://www.forexfactory.com/calendar")
        human_delay(3.0, 5.0)

        for event in events:
            print(f"\n[+] Procesando: {event}")

            # Recupera la última fecha guardada para este evento
            with engine.connect() as conn:
                result = conn.execute(
                    text('SELECT MAX(date) FROM Events WHERE event = :e'),
                    {"e": event}
                )
                last_date = result.scalar()
                last_date = pd.to_datetime(last_date).normalize() if last_date else None

            try:
                driver = select_event(event, driver)
                driver = load_all_history(driver)
                data   = parse_events(driver)
            except (TimeoutException, NoSuchElementException) as e:
                print(f"  ⚠ Error al procesar '{event}': {e}")
                # Recarga el calendario y continúa con el siguiente
                driver.get("https://www.forexfactory.com/calendar")
                human_delay(3.0, 5.0)
                continue

            # Guarda solo los registros nuevos
            saved = 0
            with engine.connect() as conn:
                for date, (current, previous) in data.items():
                    if last_date is None or date > last_date:
                        conn.execute(
                            text("INSERT OR IGNORE INTO Events VALUES (:d, :e, :c, :p)"),
                            {"d": str(date), "e": event, "c": current, "p": previous}
                        )
                        saved += 1
                conn.commit()

            print(f"  ✓ {saved} nuevos registros guardados para '{event}'")

            # Vuelve al calendario para el siguiente evento
            driver.get("https://www.forexfactory.com/calendar")
            human_delay(2.5, 4.5)

    finally:
        driver.quit()
        print("\n✅ Todo actualizado. Driver cerrado.")


if __name__ == "__main__":
    search_event()


[+] Procesando: US Federal Funds Rate
  → 156 registros encontrados
  ✓ 0 nuevos registros guardados para 'US Federal Funds Rate'

[+] Procesando: US FOMC Statement
  → 156 registros encontrados
  ✓ 0 nuevos registros guardados para 'US FOMC Statement'

[+] Procesando: US FOMC Press Conference

✅ Todo actualizado. Driver cerrado.


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=145.0.7632.162); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x10e7dd3
	0x10e7e14
	0xef1db0
	0xee0d4e
	0xeff895
	0xf652ec
	0xf7b0d9
	0xf5e7d6
	0xf30049
	0xf30e04
	0x1346924
	0x1341bf7
	0x135f5a0
	0x1100f58
	0x110891d
	0x10f0648
	0x10f0812
	0x10da21a
	0x750d5d49
	0x7719d81b
	0x7719d7a1
